In [ ]:
import os
import re
import logging
from params.paths import ROOT_DIR
from datetime import datetime, timedelta
from typing import Iterator
import psycopg2
from dotenv import load_dotenv
import time
from tqdm import tqdm


REPR_SPEECHES_DIR = os.path.join(ROOT_DIR, 'data', 'data_repr_speeches')
print(os.listdir(REPR_SPEECHES_DIR))

['無所属', '自民', '国民', 'summary.json', '民主', 'Ｎ党', '公明', '立憲', 'れ新', 'diachronic_stats.json', 'LGBT_summaries.hdf5', '参政', '保守', '維新', '無', '共産', '沖縄', '有志']


In [10]:
def iterate_speech_files(root_dir: str) -> Iterator[str]:
	for party in os.listdir(root_dir):
		party_dir = os.path.join(root_dir, party)
		if not os.path.isdir(party_dir):
			continue
		for repr in os.listdir(party_dir):
			repr_dir = os.path.join(party_dir, repr)
			if not os.path.isdir(repr_dir):
				continue
			for speech_file in os.listdir(repr_dir):
				yield os.path.join(repr_dir, speech_file)



In [11]:
load_dotenv()

conn = None
try:
    conn = psycopg2.connect(
        dbname="kokkaidoc",
        user="postgres",
        password=os.getenv("PSQL_DATABASE_PASSWORD"),
        host="localhost",
        port="5432",
    )
    print("Connected.")

    with conn.cursor() as cur:

        # Insert everything in one transaction
        for hist_dir in [LOWER_HOUSE_DATA_HISTORICAL_TMP, UPPER_HOUSE_DATA_HISTORICAL_TMP]:
            for raw in iterate_over_historical_data(hist_dir):
                upsert_person_and_elections(cur, raw)

    conn.commit()
    print("Done.")

except Exception as e:
    if conn:
        conn.rollback()
    raise
finally:
    if conn:
        conn.close()

NameError: name 'psycopg2' is not defined